In [ ]:
"""
Gera dois arquivos CSV prontos para usar no projeto:

  portfolio_precos.csv  → preços e retornos diários dos 3 ativos
  cdi_historico.csv     → CDI diário (taxa livre de risco para o Sharpe)

Período: 2019-01-01 até 2024-12-31
Ativos:  ITUB4 (ação), BOVA11 (ETF Ibovespa), BBAS3 (ação banco)
O dataset consolidado (dataset_completo.csv) inclui:

  - Preços de fechamento ajustados (ITUB4, BOVA11, BBAS3)
  - Retornos percentuais diários (retorno_ITUB4, retorno_BOVA11, retorno_BBAS3)
  - CDI diário em decimal (CDI_decimal)
  - Retornos acima do CDI (retorno_acima_cdi_*) para cada ativo

"""

import pandas as pd
import yfinance as yf
from bcb import sgs
from pathlib import Path

# ── Configurações ─────────────────────────────────────────
ATIVOS   = ["ITUB4.SA", "BOVA11.SA", "BBAS3.SA"]
INICIO   = "2019-01-01"
FIM      = "2024-12-31"
SAIDA    = Path("dados")          # pasta onde os CSVs serão salvos
SAIDA.mkdir(exist_ok=True)

# ── 1. Preços históricos via yfinance ─────────────────────
print("Baixando preços históricos...")
raw = yf.download(ATIVOS, start=INICIO, end=FIM, auto_adjust=True, progress=False)

# Extrai só o preço de fechamento ajustado
fechamento = raw["Close"].copy()
fechamento.columns = ["ITUB4", "BOVA11", "BBAS3"]
fechamento.index.name = "data"

# Remove dias sem negociação (feriados, fins de semana)
fechamento = fechamento.dropna(how="all")

# Calcula retornos percentuais diários
retornos = fechamento.pct_change().dropna()
retornos.columns = ["retorno_ITUB4", "retorno_BOVA11", "retorno_BBAS3"]

# Junta preços e retornos em um único DataFrame
precos_completo = fechamento.join(retornos)

# Salva CSV de preços
caminho_precos = SAIDA / "portfolio_precos.csv"
precos_completo.to_csv(caminho_precos)
print(f"  ✓ {caminho_precos}  ({len(precos_completo)} pregões)")

# ── 2. CDI histórico via Banco Central ────────────────────
print("Baixando CDI histórico do Banco Central...")

# Série 12 = CDI diário (em % ao dia)
cdi_raw = sgs.get({"CDI_pct_dia": 12}, start=INICIO, end=FIM)
cdi_raw.index.name = "data"

# Converte de percentual para decimal (ex: 0.053% → 0.00053)
cdi_raw["CDI_decimal"] = cdi_raw["CDI_pct_dia"] / 100

# Salva CSV do CDI
caminho_cdi = SAIDA / "cdi_historico.csv"
cdi_raw.to_csv(caminho_cdi)
print(f"  ✓ {caminho_cdi}  ({len(cdi_raw)} dias)")

# ── 3. Dataset consolidado (tudo junto) ───────────────────
print("Consolidando dataset final...")

# Alinha as datas (apenas dias com pregão E CDI disponível)
consolidado = precos_completo.join(cdi_raw["CDI_decimal"], how="inner")
consolidado = consolidado.dropna()

# Adiciona colunas auxiliares úteis para o ambiente RL
consolidado["retorno_acima_cdi_ITUB4"] = (
    consolidado["retorno_ITUB4"] - consolidado["CDI_decimal"]
)
consolidado["retorno_acima_cdi_BOVA11"] = (
    consolidado["retorno_BOVA11"] - consolidado["CDI_decimal"]
)
consolidado["retorno_acima_cdi_BBAS3"] = (
    consolidado["retorno_BBAS3"] - consolidado["CDI_decimal"]
)

# Salva dataset consolidado
caminho_final = SAIDA / "dataset_completo.csv"
consolidado.to_csv(caminho_final)
print(f"  ✓ {caminho_final}  ({len(consolidado)} pregões)")

# ── 4. Divide treino / teste e salva separado ─────────────
print("Dividindo treino / teste...")

treino = consolidado["2019-01-01":"2023-12-31"]
teste  = consolidado["2024-01-01":"2024-12-31"]

treino.to_csv(SAIDA / "treino.csv")
teste.to_csv(SAIDA / "teste.csv")

print(f"  ✓ treino.csv  ({len(treino)} pregões — 2019 a 2023)")
print(f"  ✓ teste.csv   ({len(teste)} pregões — 2024)")

# ── 5. Resumo ─────────────────────────────────────────────
print("\n" + "="*55)
print("RESUMO DO DATASET")
print("="*55)
print(f"Período total : {consolidado.index[0].date()} → {consolidado.index[-1].date()}")
print(f"Pregões total : {len(consolidado)}")
print(f"Ativos        : ITUB4, BOVA11, BBAS3")
print(f"Colunas       : {list(consolidado.columns)}")
print(f"\nEstatísticas dos retornos diários:")
print(consolidado[["retorno_ITUB4","retorno_BOVA11","retorno_BBAS3"]].describe().round(6))
print("\nArquivos gerados em ./dados/")
print("  dataset_completo.csv   ← use este para começar")
print("  treino.csv             ← dados 2019-2023 para treinar")
print("  teste.csv              ← dados 2024 para avaliar")
print("  portfolio_precos.csv   ← só preços e retornos")
print("  cdi_historico.csv      ← só CDI diário")

: 